# Edge Classification for Branch Congestion (Colab + Deep GNN)
This notebook implements a Deep Graph Neural Network pipeline (5 layers + Residual Connections) for predicting congestion across the branches of the IEEE 30-bus grid.

In [ ]:
%pip install pandas==2.2.2 pandapower numpy torch torchvision torchaudio torch_geometric h5py

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset
from torch_geometric.nn import GCNConv, GATConv, SAGEConv
from torch_geometric.loader import DataLoader
import pandapower.networks as nw

### 1. PyTorch Geometric Dataset (Características Físicas Reales)
El dataset mapea características físicas (cargas y P_max, P_min de generadores) y soporta N-1 Contingencies.

In [ ]:
class IEEECongestionCSVDataset(Dataset):
    def __init__(self, csv_file):
        super().__init__()
        self.df = pd.read_csv(csv_file)
        
        net = nw.case30()
        
        self.load_buses = torch.tensor(net.load.bus.values, dtype=torch.long)
        self.gen_buses = torch.tensor(np.concatenate([net.ext_grid.bus.values, net.gen.bus.values]), dtype=torch.long)
        
        self.branches = []
        for idx, row in net.line.iterrows():
            self.branches.append((int(row.from_bus), int(row.to_bus)))
            
        self.branch_u = torch.tensor([b[0] for b in self.branches], dtype=torch.long)
        self.branch_v = torch.tensor([b[1] for b in self.branches], dtype=torch.long)
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx].values
        
        pd_data = torch.tensor(row[0:21], dtype=torch.float32)
        qd_data = torch.tensor(row[21:42], dtype=torch.float32)
        gen_status_data = torch.tensor(row[42:48], dtype=torch.float32)
        rmax_data = torch.tensor(row[48:54], dtype=torch.float32)
        rmin_data = torch.tensor(row[54:60], dtype=torch.float32)
        status_data = torch.tensor(row[60:101], dtype=torch.float32)
        targets = torch.tensor(row[101:], dtype=torch.float32).unsqueeze(-1)
        
        x = torch.zeros((30, 5), dtype=torch.float32)
        
        num_loads = min(len(self.load_buses), pd_data.shape[0])
        x[self.load_buses[:num_loads], 0] = pd_data[:num_loads]
        x[self.load_buses[:num_loads], 1] = qd_data[:num_loads]
        
        num_gens = min(len(self.gen_buses), gen_status_data.shape[0])
        x[self.gen_buses[:num_gens], 2] = gen_status_data[:num_gens]
        x[self.gen_buses[:num_gens], 3] = rmax_data[:num_gens]
        x[self.gen_buses[:num_gens], 4] = rmin_data[:num_gens]
        
        active_edges = []
        for branch_idx, is_active in enumerate(status_data):
            if is_active > 0.5:
                u, v = self.branches[branch_idx]
                active_edges.append([u, v])
                active_edges.append([v, u])
                
        if len(active_edges) > 0:
            edge_index = torch.tensor(active_edges, dtype=torch.long).t().contiguous()
        else:
            edge_index = torch.zeros((2, 0), dtype=torch.long) 
            
        data = Data(
            x=x,
            edge_index=edge_index,
            y=targets,
            status=status_data.unsqueeze(-1),
            num_nodes=30
        )
        return data

### 2. Deep Graph ML Model (Over-smoothing Prevention)
Para evitar que la señal se diluya en la profundidad (Over-smoothing), se usa una arquitectura profunda (5 saltos) con BatchNorm1d y Conexiones Residuales (Skip Connections).

In [ ]:
class DeepEdgeCongestionGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_layers, branch_u, branch_v):
        super().__init__()
        self.num_layers = num_layers
        
        # Proyección inicial de atributos
        self.node_encoder = nn.Linear(in_channels, hidden_channels)
        
        # Capas y normalizaciones para redes profundas
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        
        for _ in range(num_layers):
            self.convs.append(GATConv(hidden_channels, hidden_channels, heads=4, concat=False, dropout=0.1))    
            self.norms.append(nn.BatchNorm1d(hidden_channels))
        
        self.branch_u = branch_u
        self.branch_v = branch_v
        
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_channels, 1)
        )

    def forward(self, batch):
        x, edge_index = batch.x, batch.edge_index
        
        # 1. Proyección lineal inicial
        x = self.node_encoder(x)
        
        # 2. Iteración profunda con Conexiones Residuales
        for i in range(self.num_layers):
            x_res = x  # Skip connection guardada
            x = self.convs[i](x, edge_index)
            x = self.norms[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=0.2, training=self.training)
            x = x + x_res  # Sumamos la señal original
        
        # 3. Predictor de Bordes o Ramas
        num_graphs = batch.num_graphs
        num_nodes_per_graph = 30
        offsets = torch.arange(0, num_graphs * num_nodes_per_graph, num_nodes_per_graph, device=x.device)
        
        u_indices = (self.branch_u.unsqueeze(0).to(x.device) + offsets.unsqueeze(1)).view(-1)
        v_indices = (self.branch_v.unsqueeze(0).to(x.device) + offsets.unsqueeze(1)).view(-1)
        
        nodes_u = x[u_indices]
        nodes_v = x[v_indices]
        
        edge_h = torch.cat([nodes_u, nodes_v], dim=-1)
        out = self.edge_mlp(edge_h)
        return out

### 3. Execution & Training Loop

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
csv_path = "/content/drive/MyDrive/congestion_dataset_v3.csv"

dataset = IEEECongestionCSVDataset(csv_path)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = DeepEdgeCongestionGNN(
    in_channels=5, 
    hidden_channels=64, 
    num_layers=5, 
    branch_u=dataset.branch_u,
    branch_v=dataset.branch_v
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
pos_weight = torch.tensor([10.0]).to(device) 
criterion = nn.BCEWithLogitsLoss(reduction='none', pos_weight=pos_weight)
epochs = 30

with open("training_log.txt", "w") as log_file:
    log_file.write("Epoch,TrainLoss,TestAcc\n")

print(f"Executing {epochs} Epochs on {device}...")

Executing 20 Epochs on cpu...


In [6]:
for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        out = model(batch)
        y = batch.y.view(-1, 1)
        status_mask = batch.status.view(-1, 1)
        
        loss_components = criterion(out, y)
        loss_masked = (loss_components * status_mask).sum() / (status_mask.sum() + 1e-8)
        
        loss_masked.backward()
        optimizer.step()
        total_loss += loss_masked.item() * batch.num_graphs
        
    train_loss = total_loss / len(train_dataset)
    
    # Validation Phase
    model.eval()
    correct = 0
    total_active_branches = 0
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            out = model(batch)
            y = batch.y.view(-1, 1)
            status_mask = batch.status.view(-1, 1)
            
            preds = (out > 0).float()
            correct_preds = (preds == y).float() * status_mask
            
            correct += correct_preds.sum().item()
            total_active_branches += status_mask.sum().item()
            
    test_acc = correct / total_active_branches
    log_str = f"Epoch {epoch+1:02d} | Train Loss (Masked): {train_loss:.4f} | Test Acc (Masked): {test_acc:.4f}"
    print(log_str)
    
    with open("training_log.txt", "a") as log_file:
        log_file.write(f"{epoch+1},{train_loss:.4f},{test_acc:.4f}\n")

Epoch 01 | Train Loss (Masked): 0.7432 | Test Acc (Masked): 0.9204
Epoch 02 | Train Loss (Masked): 0.3501 | Test Acc (Masked): 0.6277
Epoch 03 | Train Loss (Masked): 0.3025 | Test Acc (Masked): 0.8478
Epoch 04 | Train Loss (Masked): 0.2861 | Test Acc (Masked): 0.9340
Epoch 05 | Train Loss (Masked): 0.2651 | Test Acc (Masked): 0.8405
Epoch 06 | Train Loss (Masked): 0.2572 | Test Acc (Masked): 0.9016
Epoch 07 | Train Loss (Masked): 0.2386 | Test Acc (Masked): 0.7943
Epoch 08 | Train Loss (Masked): 0.2372 | Test Acc (Masked): 0.9280
Epoch 09 | Train Loss (Masked): 0.2355 | Test Acc (Masked): 0.8342
Epoch 10 | Train Loss (Masked): 0.2261 | Test Acc (Masked): 0.9178
Epoch 11 | Train Loss (Masked): 0.2073 | Test Acc (Masked): 0.8057
Epoch 12 | Train Loss (Masked): 0.2142 | Test Acc (Masked): 0.8896
Epoch 13 | Train Loss (Masked): 0.2080 | Test Acc (Masked): 0.7670
Epoch 14 | Train Loss (Masked): 0.2201 | Test Acc (Masked): 0.8756
Epoch 15 | Train Loss (Masked): 0.2008 | Test Acc (Masked): 0.